In [32]:
%load_ext autoreload
%autoreload 2
import warnings
from pandas.errors import SettingWithCopyWarning

warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import json5,json
import fitz #type: ignore

from app.logger import *
from app.amc.fund_data import *
from app.insur.fund_data import *
from app.utils import *
from app.konstant import get_config, get_regex

utils = Helper()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [33]:
#INSURANCE FUND
amc_id = '89_0'
path = r"89_30-Apr-26_IF.pdf"
config = get_config("2026",amc_id)
regex = get_regex("2026")

object = ParmericaLifeINSR(config,regex,path)
title,path_pdf= object.check_and_highlight(path)
# print("done")
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\rep_fsparse\config\2026\89_0_AMC.json5


In [ ]:
title

In [ ]:
# object = GeneraliLifeINSR(config,regex,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [31]:
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
  
# with open("data.json","w+") as file:
#     json.dump(final_text,file)
    
# save_path = os.path.join(object.JSON_PATH, object.FILE_NAME).replace(".pdf", ".json")
# with open(save_path, 'w') as f:
#   json.dump(dfs, f, indent=2)
    
# print(f"File Saved At: {save_path}")

In [ ]:
pattern = "Rs.\\s*([\\d,.]+)"
for fund, content in final_text.items():
    # check = 'before.fund_manager'
    for key in content:
        if key.endswith(".aum"):
            print(fund)
            text =re.sub("[^A-Za-z0-9\\s\\-\\(\\)\\.\\,\\+\\%\\:\\&]+", "",content[key]).strip()
            print(text)
            match = re.findall(pattern,text, re.IGNORECASE)
            print(match)

In [23]:
import json
import csv
from typing import Union, Dict, Any, List


def json_to_csv(input_data: str,output_file: str,spacing: int = 2) -> int:
    """
    Convert mutual fund JSON into a structured CSV.
    Parameters:
        output_file (str): Path to output CSV file
        spacing (int): Number of empty rows after each mutual fund
    Returns:
        int: Total number of rows written (excluding header)
    Raises:
        ValueError: If input data format is invalid
        IOError: If file operations fail
    """
    try:
        with open(input_data, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception:
        raise

    # Validate
    if "records" not in data or not isinstance(data["records"], list):
        raise ValueError("Invalid JSON structure")

    row_count = 0
    try:
        with open(output_file, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            # Header
            writer.writerow([
                "page",
                "table",
                "sfin",
                "mutual_fund_name",
                "main_scheme_name",
                "portfolio_data_0",
                "portfolio_data_1",
                "monthly_aum_value",
                "empty_col",
            ])

            # Process each record
            for record in data["records"]:
                value = record.get("value", {})
                mf_name = value.get("mutual_fund_name", "")
                scheme_name = value.get("main_scheme_name", "")
                aum_value = value.get("monthly_aaum_value", "")
                portfolio_list: List[Dict[str, Any]] = value.get("portfolio_data", [])
                sfin = value.get("sfin","")

                # Skip if no portfolio data
                if not isinstance(portfolio_list, list):
                    continue

                for item in portfolio_list:
                    writer.writerow([
                        item.get("page", ""),
                        item.get("table", ""),
                        sfin,
                        mf_name,
                        scheme_name,
                        item.get("0", ""),
                        item.get("1", ""),
                        aum_value,
                        "",
                    ])
                    row_count += 1

                # Add spacing rows
                for _ in range(spacing):
                    writer.writerow([])
        return row_count
    except Exception as e:
        raise IOError(f"Error writing CSV: {e}")

In [24]:
path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\81_30-Apr-26_IF.json"
# path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\78_30-Apr-26_IF.json" #AXA
# path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\74_30-Apr-26_IF.json" #Bandhan
path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\94_30-Apr-26_IF.json"
from pathlib import Path


_path_ = Path(path)
output_path = _path_.name.replace(".json",".csv")
json_to_csv(path,output_path)

3580